# The workbench

This notebook walks the live devising path one call at a time, and shows everything each call sends and everything it gets back. It holds no copy of anything: it reads the prompts out of `agents/` and `shared/`, and it makes the calls `panel/devising.py` and `panel/continuing.py` make.

The product repairs, checks and screens without saying so. Each of those has a cell here, in the order the container runs them.

One activity is one activity. Decide here what to try; `python -m research.run` answers whether it worked.

In [1]:
import os
import secrets
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import research.bench as bench  # noqa: E402
from agents.experience_deviser import ExperienceDeviser, experience_in, the_prompt  # noqa: E402
from orchestrator.router import FoundryConfig, FoundryRouter  # noqa: E402
from orchestrator.safety import (  # noqa: E402
    AzureContentSafetyGate,
    ContentSafetyConfig,
    screen_experience,
)
from shared.agents import AgentContext  # noqa: E402
from shared.ids import LearnerId  # noqa: E402
from shared.seal import Sealer, SealPurpose  # noqa: E402

bench.environment()

# Built exactly as `panel/devising.py` builds it, so the gate and the router here are the
# ones production uses. The safety key is per-process: the seal this gate mints travels
# nowhere, on this path or on that one.
ENVIRONMENT = dict(os.environ)
KEY = ENVIRONMENT.get("LANTERNINA_SAFETY_KEY", "").encode() or secrets.token_bytes(32)
GATE = AzureContentSafetyGate(
    ContentSafetyConfig.from_env(ENVIRONMENT),
    Sealer(SealPurpose.CONTENT_SAFETY, KEY, "orchestrator.safety"),
)
ROUTER = FoundryRouter(FoundryConfig.from_env(ENVIRONMENT), gate=GATE)
CTX = AgentContext(router=ROUTER, learner_id=LearnerId(""), learner_hints={}, now=time.time())
DEVISER = ExperienceDeviser()

print("prompt fingerprint:", bench.reload_prompts())

# Every call any router makes, with the prompt as it went and the answer as it came.
# Patched on the class and not on CTX's router: `panel/continuing.py` builds its own, and a
# call made inside an agent would otherwise leave no trace.
CALLS: list[dict[str, object]] = []


def _remember(name: str):
    original = getattr(FoundryRouter, name)

    async def instead(self, request):
        began = time.time()
        payload = await original(self, request)
        came = getattr(payload, "text", None)
        if came is None:
            came = f"<{len(getattr(payload, 'body', b''))} bytes of image>"
        CALLS.append(
            {
                "purpose": request.purpose or name,
                "seconds": round(time.time() - began, 1),
                "sent": request.prompt,
                "came": came,
            }
        )
        print(
            f"[{len(CALLS)}] {request.purpose} · {CALLS[-1]['seconds']} s · "
            f"{len(request.prompt)} characters sent, {len(came)} back"
        )
        return payload

    setattr(FoundryRouter, name, instead)


if not getattr(FoundryRouter, "_remembered", False):
    for _name in ("analyze", "generate_for_user"):
        _remember(_name)
    FoundryRouter._remembered = True

BAR = "=" * 78


def said(n: int = -1) -> None:
    """One call whole: what went up and what came back."""
    one = CALLS[n]
    print(f"{BAR}\n[{n if n >= 0 else len(CALLS) - 1}] {one['purpose']} · {one['seconds']} s")
    print(f"{BAR}\nWHAT WAS SENT — {len(one['sent'])} characters\n{BAR}")
    print(one["sent"])
    print(f"{BAR}\nWHAT CAME BACK — {len(one['came'])} characters\n{BAR}")
    print(one["came"])


def conversation(to: Path | None = None) -> Path:
    """Every call so far, in order, whole, written where it can be read and searched."""
    to = to or ROOT / "tmp" / "conversation.txt"
    to.parent.mkdir(exist_ok=True)
    parts = []
    for n, one in enumerate(CALLS):
        parts.append(f"{BAR}\n[{n}] {one['purpose']} · {one['seconds']} s\n{BAR}")
        parts.append(f"--- SENT ({len(one['sent'])} characters) ---\n{one['sent']}")
        parts.append(f"--- CAME BACK ({len(one['came'])} characters) ---\n{one['came']}\n")
    to.write_text("\n".join(parts), encoding="utf-8", newline="")
    for n, one in enumerate(CALLS):
        print(f"[{n}] {one['purpose']:44} {one['seconds']:6} s  "
              f"{len(one['sent']):7} sent  {len(one['came']):7} back")
    print(f"\n{len(CALLS)} calls · {to}")
    return to


prompt fingerprint: 4358f1f9c9ec


## The calls, in order

The container makes six calls. `panel/devising.py` makes 1 to 4 inside one request and then returns. The house walks the moments, and call 5 happens at every `hand_over`. `panel/continuing.py` makes call 6 when a page comes back off the glass and an outcome says `ask`.

1. **which method** — to the model that writes. Sends `choosing` and a sample of sixty method names, without the records. Comes back with two ids and a reason. A few seconds.
2. **the activity** — to the model that writes. Sends the whole standing instruction and this household's material, about 34 000 characters. Comes back with one JSON document: title, overview, themes, script, minutes, drawn, moments. 76–184 s, median about 140.
3. **a repair** — to the model that writes, and only when something has refused the answer. Sends `repair`, the refusal and the document as it was; comes back with the same JSON again. It runs once and no more. `repair_unreadable` runs when the format cannot read the answer at all, `repair` when one of the seven checks refuses it.
4. **the gate** — to the safety gate. Sends the document's own words, comes back through or as `SafetyBlocked`. Seconds.
5. **the page** — to the model that draws, at every `hand_over`. Sends `page_maker.*`, built from the page's own words. Comes back with one PNG. 25–35 s a page.
6. **the continuation** — to the model that writes, when an outcome says `ask`. Sends `experience_continuer.*`, the activity so far and what came back on the sheet. Comes back with the rest of the moments. 15–25 s.

The list names roles and not deployments. The router decides which model answers each call, and `research/env.ps1` says which one is deployed today.

The setup cell records every call on `FoundryRouter` itself, so a call made inside an agent is recorded too. `said()` prints the last call whole, `said(2)` prints the third, and `conversation()` writes them all in order to `tmp/conversation.txt`. Read that file to find the place a prompt goes wrong.

The stand-in is the one call here that the house never makes. It is `research/calls.adolescent.md`, and it lets an activity be walked with nobody in the room.

This notebook differs from the container in four ways, and none of them changes what is sent. A failed choosing call raises here, while the container draws a method instead. A document the checks still refuse after its one repair raises `RefusedByTheChecks` in the container, and here it is only printed. The gate is not closed at the end. And the walk — the stand-in, and stepping through the moments — belongs to `research/` and not to the product: in a house a person does that.

`env.ps1` holds no credentials. The router builds a `DefaultAzureCredential`, which finds the `az` session in the `AZURE_CONFIG_DIR` named there. When a call fails on authentication, run `az account get-access-token` in a terminal with that variable set; `az account show` reads the cache and succeeds even on an expired token.

## 1. The prompts, as they are

Every block, with the format's own numbers filled in. The placeholders still written `$name` carry a household's material, and the cell after next fills them.

In [2]:
P = bench.everything()

for key, text in P.items():
    print(f"{key:26} {len(text):6}")
print(f"{'':26} {sum(len(one) for one in P.values()):6}  in all")

task                         2072
format                       4929
shape-of-a-moment            2762
acts                         1098
marks-on-a-page              3463
ten-dimensions               1132
rules-head                     88
limits                        482
rules-tail                    711
asking                        374
manner-head                   389
how-the-text-reads            860
what-to-refuse                421
only-what-you-can-answer     1398
worth-doing                  5107
manner-tail                   114
method                        807
household                    1387
pitch                         513
stand-in                     1531
                            29638  in all


In [3]:
print(P["task"])

You are making one activity for one adolescent to spend at home, mostly on their own, in a house with a printer, a scanner and a small screen or two. Not a lesson, not a test, and not an exercise with a story painted over it. One thing worth doing, that happens once, and is over when it is over.

A parent glances at your overview before this happens, and may add something of their own. That is not a tribunal and you are not defending anything: they are seeing roughly what is coming, for somebody they know and you do not. Write the overview so that one glance says what this activity actually is. Nothing here is set in stone either — an activity can turn out differently once it has started, and that is ordinary rather than a failure.

An activity is a good one when four things are true of it.

Somebody can start it alone. The first screen puts a situation in front of them, they can see what to pick up, and no adult has to explain anything first.

Something is genuinely not known, and by 

## 2. The household

Everything that decides one activity, as one dictionary. `research/run.py` sends the same one to `devise_experience`.

In [4]:
from research.households import Household, Memory, arguments

HOUSE = Household(
    name="bench",
    interests=("i treni", "le mappe vecchie"),
    avoid=("i ragni",),
    load="middle",
    ink="middle",
    span="middle",
    sheets=2,
    note="",
)
MEMORY = Memory()  # no history: the first activity of this house
ARGS = arguments(HOUSE, MEMORY)

for name, value in ARGS.items():
    shown = value if isinstance(value, str) else repr(value)
    print(f"{name:12} {shown[:110] or '—'}")

capabilities frozenset({<HouseCapability.SCAN_A4: 'scan_a4'>, <HouseCapability.PRINT_A4: 'print_a4'>, <HouseCapability.SHOW
language     Italian
interests    ('i treni', 'le mappe vecchie')
avoid        ('i ragni',)
pitch        two things that have to be put side by side before either makes sense, and one turn where what looked true sto
sheets       2
note         —
already      ()
recent       ()
happened     —
counts       {"afternoonsRun": 0, "ranToTheEnd": 0, "endedEarly": 0, "stopped": 0, "sheetsWrittenOn": 0, "sheetsBlank": 0}
direction    Ask for about what the last ones asked for. Nothing here says to move either way.
ground       —


## 3. Call one: which method

This is the cheap call before the expensive one. It carries a sample of sixty names out of `methods/` and none of the records, so the corpus stays outside the prompt. The sample exists because the whole catalogue made the model pick the same pair twice out of twice: the same judgement on the same list gives the same answer, which was worse than drawing at random.

In [5]:
from shared.methods import CATALOGUE, by_id, draw, index, load, runnable

RUNNABLE = runnable(load(), capabilities=ARGS["capabilities"])
CATALOGUE_SENT = index(RUNNABLE, sample=CATALOGUE)
print(f"{len(RUNNABLE)} methods this house can run · {len(CATALOGUE_SENT)} characters of names\n")
print(CATALOGUE_SENT[:1200])

145 methods this house can run · 5102 characters of names

cut-where-the-counting-stops: cut the printed part where the counting stops (a move)
three-sheets-that-never-say-they-are-one: three sheets on three days that never say they are one
a-module-from-an-office-that-does-not-exist: a form from an office that does not exist
what-you-weighed-becomes-the-weight: weigh in coins until what you weighed is a weight
five-unalike-tasks-and-the-one-dropped-last: offer five unalike tasks and ask which was dropped (a move)
wrong-attempts-kept-side-by-side: keep the wrong attempts and read them together (a move)
open-every-task-and-ask-for-four-of-six: open every task at once and ask for four of the six (a move)
blank-at-the-end-of-the-line: leave the blank at the end of every line
the-story-stops-where-somebody-decided: stop the story where somebody had to decide
an-alphabet-tested-in-the-dark: invent an alphabet and test it in the dark
facts-printed-three-explanations: print every fact, then a

In [6]:
began = time.time()
WANTS_FORM, WANTS_MOVE, WHY = await DEVISER.choose(
    CTX,
    catalogue=CATALOGUE_SENT,
    interests=ARGS["interests"],
    avoid=ARGS["avoid"],
    already=ARGS["already"],
    pitch=ARGS["pitch"],
)
ASKED_FOR = by_id(RUNNABLE, [WANTS_FORM, WANTS_MOVE])
FORM = next((one for one in ASKED_FOR if not one.is_a_move), None)
MOVE = next((one for one in ASKED_FOR if one.is_a_move), None)
if FORM is None or MOVE is None:
    # What `panel/devising.py` does: a step that exists to improve an activity may never
    # be the step that costs one.
    FORM, MOVE = draw(RUNNABLE)
    print("the choice did not resolve; drew instead")
print(f"{time.time() - began:.1f} s · {FORM.method_id} + {MOVE.method_id}\n{WHY}")

[1] choosing what to build an afternoon out of · 16.4 s · 6521 characters sent, 198 back
16.4 s · a-machine-made-of-two-printed-tables + where-it-went-wrong-not-whether
Two old-railway tables work only together, until one route breaks and the task becomes finding where.


In [7]:
said()

[0] choosing what to build an afternoon out of · 16.4 s
WHAT WAS SENT — 6521 characters
You are about to devise one activity for one adolescent to spend at home. First you are choosing what to build it out of.

Below is a catalogue of methods that work on paper, filtered to the ones this house can actually run. Each line is an id and a name. Some are marked as a move: a move is not something to do on its own — it is applied to a form and changes what the form asks of somebody.

The parent wrote down these interests, as a place to begin: ["i treni", "le mappe vecchie"]
And these things to keep away from: ["i ragni"]
Activities already offered here: []
How the activity should be pitched: two things that have to be put side by side before either makes sense, and one turn where what looked true stops being true
a sheet may ask for a few words as well as marks, and the space left for them should be small enough that filling it is obviously finishable
write the standard weight at the middle 

## 4. The whole prompt, before paying for it

In [8]:
PROMPT = the_prompt(**ARGS, form=FORM, move=MOVE)
print(f"{len(PROMPT)} characters\n")
print(PROMPT)

34038 characters

You are making one activity for one adolescent to spend at home, mostly on their own, in a house with a printer, a scanner and a small screen or two. Not a lesson, not a test, and not an exercise with a story painted over it. One thing worth doing, that happens once, and is over when it is over.

A parent glances at your overview before this happens, and may add something of their own. That is not a tribunal and you are not defending anything: they are seeing roughly what is coming, for somebody they know and you do not. Write the overview so that one glance says what this activity actually is. Nothing here is set in stone either — an activity can turn out differently once it has started, and that is ordinary rather than a failure.

An activity is a good one when four things are true of it.

Somebody can start it alone. The first screen puts a situation in front of them, they can see what to pick up, and no adult has to explain anything first.

Something is genuinely 

## 5. Call two: the activity

One answer, whole. Measured at 76–184 s, median about 140.

In [9]:
began = time.time()
ANSWER = await DEVISER.ask(CTX, **ARGS, form=FORM, move=MOVE)
print(f"{time.time() - began:.1f} s · {len(ANSWER)} characters back")
print(ROUTER.last_usage)

[2] devising an afternoon · 136.8 s · 34038 characters sent, 11010 back
136.8 s · 11010 characters back
ModelUsage(deployment='gpt-5.6-sol-2026-07-09', request_id='ca87b4dc-8a8c-4091-a565-9de853f14e18', input_tokens=7827, output_tokens=8317, cached_input_tokens=0, reasoning_tokens=5091, size='', quality='')


In [10]:
said()

[1] devising an afternoon · 136.8 s
WHAT WAS SENT — 34038 characters
You are making one activity for one adolescent to spend at home, mostly on their own, in a house with a printer, a scanner and a small screen or two. Not a lesson, not a test, and not an exercise with a story painted over it. One thing worth doing, that happens once, and is over when it is over.

A parent glances at your overview before this happens, and may add something of their own. That is not a tribunal and you are not defending anything: they are seeing roughly what is coming, for somebody they know and you do not. Write the overview so that one glance says what this activity actually is. Nothing here is set in stone either — an activity can turn out differently once it has started, and that is ordinary rather than a failure.

An activity is a good one when four things are true of it.

Somebody can start it alone. The first screen puts a situation in front of them, they can see what to pick up, and no adult has 

## 6. The format reads it

In [11]:
from shared.experience import ExperienceError

REFUSAL = ""
try:
    EXPERIENCE = experience_in(ANSWER)
except ExperienceError as exc:
    EXPERIENCE, REFUSAL = None, str(exc)
    print("the format would not read it:", REFUSAL)
else:
    print(f"{EXPERIENCE.title} · {len(EXPERIENCE.moments)} moments · {EXPERIENCE.minutes} min")

Gli sportelli della stazione di Valnera · 5 moments · 95 min


If it would not read it at all, the answer goes back up with the refusal. `repair_unreadable`.

In [12]:
if REFUSAL:
    began = time.time()
    EXPERIENCE = await DEVISER.repair_unreadable(
        CTX, answer=ANSWER, refusal=REFUSAL, language=ARGS["language"]
    )
    print(f"{time.time() - began:.1f} s · {EXPERIENCE.title}")
else:
    print("the format read it; nothing to repair")

the format read it; nothing to repair


## 7. The seven checks

`shared/experience_checks.py` reads the shape of the document and not its sense. A complaint that comes back every activity names the rule in the prompt to work on.

In [13]:
from shared.experience_checks import check

COMPLAINTS = check(EXPERIENCE, recent=ARGS["recent"], sheets_at_most=ARGS["sheets"])
print("; ".join(map(str, COMPLAINTS)) or "no complaints")

no complaints


One repair and no more, which is what `panel/devising.py` allows.

In [14]:
if COMPLAINTS:
    began = time.time()
    EXPERIENCE = await DEVISER.repair(
        CTX, refused=EXPERIENCE, complaints=COMPLAINTS, language=ARGS["language"]
    )
    COMPLAINTS = check(EXPERIENCE, recent=ARGS["recent"], sheets_at_most=ARGS["sheets"])
    print(f"{time.time() - began:.1f} s · after the repair: ")
    print("; ".join(map(str, COMPLAINTS)) or "no complaints")
else:
    print("nothing to repair")

nothing to repair


## 8. The gate

This is the chokepoint. Nothing on this path returns past it: the parse and the checks can refuse a document, and only screening lets one through.

In [15]:
began = time.time()
await screen_experience(GATE, EXPERIENCE, context="devising an activity")
DOCUMENT = EXPERIENCE.to_dict()
print(f"{time.time() - began:.1f} s · the gate let it through")

1.7 s · the gate let it through


## 9. What came out

Only what would reach somebody in the room.

In [16]:
from tools.as_it_arrives import read

COUNTED = read(DOCUMENT)

──────────────────────────────────────────────────────────────────────────────
  Gli sportelli della stazione di Valnera
  Confronta una griglia di instradamento con una pianta ferroviaria, prova coppie di numeri e segna dove una regola apparente cede. Dalla stampante escono il quaderno di un’instradatrice e una vecchia pianta degli sportelli. Alla fine resta un’annotazione propria nel quaderno. Non serve altro. Può non piacere che due treni sembrino diretti verso una collisione mai avvenuta.
  95 minuti
──────────────────────────────────────────────────────────────────────────────

1. [say] Una pagina nel vassoio
    Valnera riceve due treni insieme.
    Lidia decideva quale uscita usare.
    Prendi la matita dal tavolo.
      ↳ dopo 1 min: La matita basta per entrare.
      ↳ dopo 3 min: Il quaderno arriverà nel vassoio.
      ↳ dopo 6 min: Le coppie vanno provate su carta.
      ↳ dopo 9 min: Il custode apre il quaderno di Lidia.
      ⇥ via d'uscita, con «matita»: La matita resta



And the script, which is what the parent approves and what whoever runs the activity reads.

In [17]:
print(EXPERIENCE.script)

IL MONDO La stazione di Valnera riceve due treni alla volta, distinti solo da un numero. La luce filtra da vetri color fumo e l’aria sa di ferro bagnato. Nessuno nomina la galleria sotto la sala d’attesa. LA VIA D’INGRESSO Nel vassoio della stampante compare oggi una pagina del quaderno di Lidia, l’ultima instradatrice. Sul tavolo c’è una matita. La pagina manda ogni coppia di treni a uno sportello, ma il motivo dell’ultima modifica non è scritto. LA DOMANDA Risposta: Lidia temeva che due treni con lo stesso numero entrassero insieme, da lati opposti, nella galleria a binario unico. Cosa la stabilisce: la griglia delle coppie; l’elenco separato degli sportelli; le uscite 9 e 16 che convergono sulla pianta. Operazione: confronto. Falsa risposta: Lidia cercava sempre il percorso più corto; crolla perché la pianta dice che le uscite scelte, 6 e 8, sono più lunghe di 9 e 16. Chi voleva saperlo: il nuovo custode di Valnera, perché deve decidere se conservare la modifica di Lidia. I BATTITI 

## 10. The walk

Run the four cells below in order, then run them again, until the activity is over. Each pass is one moment.

⚠️ The image deployment has capacity 2 in this region, so two page calls within about 30 s answer 429. A page takes 25–35 s and costs a few cents. `research/env.ps1` says which deployment that is, and the cell above prints what the last call cost.

In [18]:
from IPython.display import Image, display

from agents.page_maker import PageMaker
from shared.experience import ASK, Collect, HandOver, Say, Weight

WEIGHT = Weight.STANDARD
MOOD = "una giornata normale, c'è voglia di fare qualcosa"

BY_ID = {one.id: one for one in EXPERIENCE.moments}
ORDER = [one.id for one in EXPERIENCE.moments]
AT = ORDER[0]
MINUTES = 0
DISPLAYS: list[str] = []
SHEET = ""
CAME = None


def show(moment) -> None:
    """What this moment puts in the room, and the ladder somebody stuck would meet."""
    global MINUTES, SHEET
    weighing = moment.at(WEIGHT)
    MINUTES += weighing.minutes
    print(f"[{moment.act}] {moment.heading}   ({weighing.minutes} min, {MINUTES} in)")
    for line in weighing.lines:
        print(f"   display: {line}")
        DISPLAYS.append(line)
    for rung in moment.help:
        print(f"     after {rung.after_minutes} min: {' '.join(rung.lines)}")
    if isinstance(moment, HandOver):
        page = moment.page
        SHEET = "\n".join(
            [f"[{page.kind}] {page.title}", *page.note]
            + [f"- {one.label} ({one.room})" for one in page.spaces]
        )
        print(f"\n   page [{page.kind}] {page.title}")
        for line in page.note:
            print(f"     {line}")
        for space in page.spaces:
            print(f"     [ {space.label} — {space.room} ]")
        print(f"     drawing: {page.illustration}")
    if isinstance(moment, Collect):
        print(f"   way out: {moment.way_out.heading} ({moment.way_out.in_hand})")
        for one in moment.outcomes:
            print(f"   if {one.when} -> {one.then}")


def next_id(moment, came: str | None = None) -> str | None:
    """Where the activity goes after this. None is an ending, ASK is a continuation."""
    if isinstance(moment, (Say, HandOver)):
        i = ORDER.index(moment.id)
        return ORDER[i + 1] if i + 1 < len(ORDER) else None
    if isinstance(moment, Collect):
        return next((one.then for one in moment.outcomes if str(one.when) == came), ASK)
    return None

**a.** Where we are.

In [19]:
M = BY_ID[AT]
show(M)

[say] Una pagina nel vassoio   (7 min, 7 in)
   display: Valnera riceve due treni insieme.
   display: Lidia decideva quale uscita usare.
   display: Prendi la matita dal tavolo.
     after 1 min: La matita basta per entrare.
     after 3 min: Il quaderno arriverà nel vassoio.
     after 6 min: Le coppie vanno provate su carta.
     after 9 min: Il custode apre il quaderno di Lidia.


**b.** The page, if this moment hands one over.

In [20]:
if isinstance(M, HandOver):
    began = time.time()
    PNG, ASKED = await PageMaker().draw(CTX, M.page)
    (ROOT / "tmp").mkdir(exist_ok=True)
    (ROOT / "tmp" / f"{M.id}.png").write_bytes(PNG)
    print(f"{time.time() - began:.1f} s · {len(PNG)} bytes")
    display(Image(PNG, width=520))
    print(ASKED)
else:
    print(f"{M.act} — nothing to draw here")

say — nothing to draw here


**c.** What the person did with it, if this moment collects one.

In [21]:
from string import Template

from research.calls import what_they_did

if isinstance(M, Collect):
    print(Template(P["stand-in"]).substitute(
        displays="\n".join(DISPLAYS), sheet=SHEET, mood=MOOD, minutes=MINUTES
    ))
    DID = await what_they_did(
        CTX, displays=DISPLAYS, sheet=SHEET, mood=MOOD, minutes_in=MINUTES
    )
    CAME = "marks" if str(DID.get("came")) == "marks" else "blank"
    print(f"\n{CAME} · {DID.get('onIt')} · stop: {DID.get('stop')}\n{DID.get('why')}")
else:
    CAME = None
    print(f"{M.act} — nobody is being asked for anything here")

say — nobody is being asked for anything here


**d.** Take the branch, and go back to **a**.

In [22]:
AT = next_id(M, CAME)
if AT is None:
    print("over")
elif AT == ASK:
    print("the outcome says ask: the rest is bought from the continuer, in the cell below")
else:
    print(f"next: {AT}  ({BY_ID[AT].act})")

next: route-notebook  (hand_over)


**e.** And if the branch said `ask`, buy the rest. This is `panel/continuing.py`, the second prompt, with its own gate and its own checks and no repair.

In [23]:
from panel.continuing import continue_experience

if AT == ASK:
    began = time.time()
    CARRYING_ON, SPENT = await continue_experience(
        experience=DOCUMENT,
        after=M.id,
        came=CAME,
        reading={"came": CAME, "reading": str(DID.get("onIt", ""))},
        now=time.time(),
        pitch=ARGS["pitch"],
    )
    print(f"{time.time() - began:.1f} s · {len(CARRYING_ON.moments)} more moments\n")
    for one in CARRYING_ON.moments:
        print(f"{one.id:26} {one.act:10} {one.heading}")
    # Carry on walking into what was just bought.
    BY_ID |= {one.id: one for one in CARRYING_ON.moments}
    ORDER += [one.id for one in CARRYING_ON.moments]
    AT = CARRYING_ON.moments[0].id
else:
    print("no continuation was asked for")

no continuation was asked for


## 11. Changing a block

The blocks are the files in `agents/` and `shared/`. Edit one in the editor, run `bench.reload_prompts()`, and go back to section 4: the fingerprint changes only when the text being sent changed.

When one is settled: `python -m tools.prompts --write`, then `python -m pytest -q`, then `python -m research.run --iterations 4 --seed 0 --label <name>` — six households, four iterations, about an hour, and the eight axes are the answer to whether it improved.

⚠️ Do not run `pytest` and a run at the same time: `tests/test_trail.py` fails with `Event loop is closed` on contention, and it looks like a real regression.

In [24]:
print("fingerprint:", bench.reload_prompts())

fingerprint: 4358f1f9c9ec


## 12. The whole conversation

Every prompt and every answer, in order, in one file. Read it when an activity comes out wrong. What the answer did that nobody asked for usually points at the sentence that asked for it.

In [25]:
conversation()

[0] choosing what to build an afternoon out of     16.4 s     6521 sent      198 back


[1] devising an afternoon                         136.8 s    34038 sent    11010 back

2 calls · C:\code\lanternina\tmp\conversation.txt


WindowsPath('C:/code/lanternina/tmp/conversation.txt')